# Lab L3.3 · Card-fraud detection on 1.5M transactions — velocity features, decision trees and random forests
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 3 · ⏱ 80 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Engineer real-time-safe velocity features at scale, grow and prune interpretable trees, beat them with a random forest, and evaluate against the fraud team’s daily alert budget.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the hint in the HTML lab guide, or peek at the Level 3 solution.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

HGB’s rules engine ("block foreign e-commerce > AED 5,000") catches too little fraud and annoys customers. The Fraud Operations team can manually review **0.5% of transactions** per day. Your model must rank transactions so that the reviewed 0.5% contains as much fraud — and as many dirhams of fraud — as possible, and its logic must be explainable to auditors.

**Progressive architecture:** your feature table feeds the SVM and neural-network labs (Day 4); your feature function is deployed and monitored on Day 5.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load 1.5M transactions efficiently and split in time

**Why this matters.** Fraud patterns evolve, so the honest split is **out-of-time**. Loading with explicit dtypes keeps memory manageable and makes `groupby` fast.

In [ ]:
tx = pd.read_csv(DATA / "card_transactions.csv.gz", parse_dates=["txn_ts"],
                 dtype={c: "category" for c in ["merchant_category", "channel", "merchant_country", "customer_home_emirate"]})
print(f"{len(tx):,} rows | {tx.memory_usage(deep=True).sum() / 1e6:,.0f} MB | fraud rate {tx.is_fraud.mean():.3%}")
print(tx.groupby("channel", observed=True).is_fraud.agg(["mean", "sum"]).sort_values("mean", ascending=False).round(4))
TRAIN_END, VALID_END = pd.Timestamp("2025-08-01"), pd.Timestamp("2025-09-01")

**✔ Checkpoint.** 1,506,430 rows; ≈ 0.43% fraud; E-Commerce and ATM have the highest fraud rates.

## Step 2 · Velocity features — the fraud analyst’s secret weapon

**Why this matters.** A single transaction looks normal; **five in ten minutes on a new device abroad** does not. Velocity features summarise each card’s recent behaviour. Every one must use only the current and **earlier** transactions of that card, otherwise it leaks the future and cannot be computed at authorisation time.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — transactions of this card in the trailing hour (incl. current)
- **Blank 2** *(inline)* — mean of the card’s EARLIER amounts only
- **Blank 3** *(code block, ≈1 lines)* — flag the first time each card is used in a given merchant_country (hint: duplicated on two columns)

In [ ]:
def build_fraud_features(tx: pd.DataFrame) -> pd.DataFrame:
    """Per-card behavioural (velocity) features using ONLY each card's past and present — safe for real-time scoring."""
    tx = tx.sort_values(["card_id", "txn_ts"]).reset_index(drop=True)
    g = tx.groupby("card_id")
    tx["secs_since_prev"] = g.txn_ts.diff().dt.total_seconds().fillna(30 * 86400)
    roll = tx.set_index("txn_ts").groupby("card_id").amount_aed
    tx["txn_count_1h"] = ____  # ✏️ B1
    r24 = roll.rolling("24h").agg(["count", "sum"])
    tx["txn_count_24h"], tx["amount_sum_24h"] = r24["count"].to_numpy(), r24["sum"].to_numpy()
    prior_mean = g.amount_aed.transform(____)  # ✏️ B2
    tx["amount_vs_card_mean"] = tx.amount_aed / prior_mean.fillna(tx.amount_aed.median())
    # ✏️ BLANK 3 — write ≈1 line(s): flag the first time each card is used in a given merchant_country (hint: duplicated on two columns)
    # ...your code here...
    tx["is_new_mcc_for_card"] = (~tx.duplicated(["card_id", "merchant_category"])).astype(int)
    tx["card_txn_index"] = g.cumcount()
    tx["hour"] = tx.txn_ts.dt.hour
    tx["is_night"] = tx.hour.between(0, 5).astype(int)
    tx["is_international"] = (tx.merchant_country != "AE").astype(int)
    tx["amount_to_salary"] = tx.amount_aed / tx.customer_monthly_salary_aed
    tx["log_amount"] = np.log1p(tx.amount_aed)
    return tx.sort_values("txn_ts").reset_index(drop=True)

NUMF = ["log_amount", "merchant_risk_score", "is_3ds_authenticated", "is_new_device", "distance_from_home_km", "customer_age",
        "card_tenure_months", "secs_since_prev", "txn_count_1h", "txn_count_24h", "amount_sum_24h", "amount_vs_card_mean",
        "is_new_country_for_card", "is_new_mcc_for_card", "card_txn_index", "hour", "is_night", "is_international", "amount_to_salary"]
CATF = ["merchant_category", "channel"]

In [ ]:
t0 = time.perf_counter()
tf = build_fraud_features(tx)
tf["split"] = np.select([tf.txn_ts < TRAIN_END, tf.txn_ts < VALID_END], ["train", "valid"], "test")
print(f"features built in {time.perf_counter() - t0:.0f}s"); print(tf.split.value_counts())
tf.groupby("is_fraud")[["txn_count_1h", "secs_since_prev", "amount_vs_card_mean", "is_new_country_for_card"]].median()

**✔ Checkpoint.** Built in ≈ 10–30 s. Fraud: median `secs_since_prev` ≈ 10 minutes vs ≈ 3 days for legit; 3 transactions in the last hour vs 1; amount ≈ 2× the card’s usual vs ≈ 0.45×.

## Step 3 · An interpretable decision tree — and its rules

**Why this matters.** Trees need no scaling and handle interactions natively. A depth-3 tree is a set of human-readable rules your auditors can review. Missing values for *not-applicable* fields (3DS on POS) are encoded as −1 so the tree can split on ‘not e-commerce’.

#### ✏️ What to write in the next cell
- **Blank 4** *(inline)* — −1 = 'not applicable' for 3DS / device / distance
- **Blank 5** *(inline)* — impurity measure used to choose splits

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import average_precision_score
prep = ColumnTransformer([("num", ____, NUMF),  # ✏️ B4
                          ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATF)], verbose_feature_names_out=False)
tr, va, te = (tf[tf.split == s] for s in ["train", "valid", "test"])
FEATS = NUMF + CATF

def at_budget(y, p, amount, budget=0.005) -> dict:
    y, amount = np.asarray(y), np.asarray(amount)
    top = np.argsort(-p)[: int(len(p) * budget)]
    return {"precision@0.5%": y[top].mean(), "recall@0.5%": y[top].sum() / y.sum(),
            "AED_caught_share": (amount[top] * y[top]).sum() / (amount * y).sum(), "PR_AUC": average_precision_score(y, p)}

board = {}
for crit in ["gini", "entropy"]:
    t3 = Pipeline([("prep", prep), ("dt", DecisionTreeClassifier(max_depth=3, criterion=____, random_state=RNG))]).fit(tr[FEATS], tr.is_fraud)  # ✏️ B5
    board[f"tree depth3 {crit}"] = at_budget(va.is_fraud, t3.predict_proba(va[FEATS])[:, 1], va.amount_aed)
plt.figure(figsize=(18, 6)); plot_tree(t3.named_steps["dt"], feature_names=FEATS, filled=True, proportion=True, fontsize=8, class_names=["legit", "fraud"]); plt.show()
print(export_text(t3.named_steps["dt"], feature_names=FEATS, decimals=2))
pd.DataFrame(board).T.round(3)

**✔ Checkpoint.** Root split on a velocity or 3DS/device feature; a depth-3 tree already captures ≈ 40–60% of fraud in the 0.5% budget. Gini and entropy give near-identical trees.

## Step 4 · Depth, leaf size and cost-complexity pruning

**Why this matters.** An unlimited tree memorises 1M training rows — perfect on train, worse on August. We sweep `max_depth`, then use **cost-complexity pruning** (`ccp_alpha`), which removes branches whose impurity reduction does not justify their complexity.

In [ ]:
Xtr, Xva = prep.fit_transform(tr[FEATS]), prep.transform(va[FEATS])
depths, rec = [2, 4, 6, 8, 12, 16, None], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, min_samples_leaf=1, random_state=RNG).fit(Xtr, tr.is_fraud)
    rec.append((str(d), average_precision_score(tr.is_fraud, dt.predict_proba(Xtr)[:, 1]), average_precision_score(va.is_fraud, dt.predict_proba(Xva)[:, 1])))
rec = pd.DataFrame(rec, columns=["max_depth", "train_PR_AUC", "valid_PR_AUC"]).set_index("max_depth")
rec.plot(marker="o", title="Deeper trees memorise: train ↑ while validation peaks"); plt.show()
best_depth = rec.valid_PR_AUC.idxmax(); best_depth = None if best_depth == "None" else int(best_depth)
dt_best = DecisionTreeClassifier(max_depth=best_depth, random_state=RNG).fit(Xtr, tr.is_fraud)
board[f"tree depth={best_depth}"] = at_budget(va.is_fraud, dt_best.predict_proba(Xva)[:, 1], va.amount_aed)

#### ✏️ What to write in the next cell
- **Blank 6** *(inline)* — all effective alphas where a subtree is pruned

In [ ]:
from sklearn.model_selection import train_test_split
Xs_, _, ys_, _ = train_test_split(Xtr, tr.is_fraud, train_size=150_000, stratify=tr.is_fraud, random_state=RNG)
path = ____  # ✏️ B6
alphas = np.quantile(path.ccp_alphas[path.ccp_alphas > 0], [0.5, 0.7, 0.8, 0.9, 0.95, 0.97, 0.99, 0.995])
pruned = [(a, DecisionTreeClassifier(ccp_alpha=a, random_state=RNG).fit(Xs_, ys_)) for a in alphas]
scores_ = [(a, m.tree_.node_count, average_precision_score(va.is_fraud, m.predict_proba(Xva)[:, 1])) for a, m in pruned]
display(pd.DataFrame(scores_, columns=["ccp_alpha", "nodes", "valid_PR_AUC"]).round(6))
a_best, m_best = max(pruned, key=lambda am: average_precision_score(va.is_fraud, am[1].predict_proba(Xva)[:, 1]))
board["tree pruned (150k)"] = at_budget(va.is_fraud, m_best.predict_proba(Xva)[:, 1], va.amount_aed)
pd.DataFrame(board).T.round(3)

**✔ Checkpoint.** Train PR-AUC → 1.0 as depth grows; validation peaks around depth 8–12 then falls. Pruning yields a much smaller tree with comparable validation PR-AUC.

## Step 5 · Random forest on an undersampled training set

**Why this matters.** A **random forest** averages many deep trees, each grown on a bootstrap sample with random feature subsets — reducing variance. With 0.4% fraud, keeping all fraud and ~25% of legitimate transactions speeds training 4× with little loss of ranking power. Undersampling inflates probabilities; ranking-based metrics (budget, PR-AUC) are unaffected, and we correct the probabilities explicitly.

#### ✏️ What to write in the next cell
- **Blank 7** *(inline)* — number of bagged trees
- **Blank 8** *(inline)* — random feature subset per split
- **Blank 9** *(code block, ≈1 lines)* — undo the undersampling bias: p = beta * p_s / (beta * p_s - p_s + 1)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
legit = tr[tr.is_fraud == 0]
tr_u = pd.concat([tr[tr.is_fraud == 1], legit.sample(250_000, random_state=RNG)])
beta = 250_000 / len(legit)                      # sampling rate of the majority class
t0 = time.perf_counter()
rf = Pipeline([("prep", prep), ("rf", RandomForestClassifier(n_estimators=____, min_samples_leaf=10, max_features=____,  # ✏️ B7; B8
                                                             n_jobs=-1, random_state=RNG))]).fit(tr_u[FEATS], tr_u.is_fraud)
print(f"RF trained on {len(tr_u):,} rows in {time.perf_counter() - t0:.0f}s")
p_rf_s = rf.predict_proba(va[FEATS])[:, 1]
board["random forest"] = at_budget(va.is_fraud, p_rf_s, va.amount_aed)
lr = Pipeline([("prep", prep), ("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=2000))]).fit(tr_u[FEATS], tr_u.is_fraud)
board["logistic (same features)"] = at_budget(va.is_fraud, lr.predict_proba(va[FEATS])[:, 1], va.amount_aed)
# ✏️ BLANK 9 — write ≈1 line(s): undo the undersampling bias: p = beta * p_s / (beta * p_s - p_s + 1)
# ...your code here...
print(f"mean predicted fraud prob — raw: {p_rf_s.mean():.4f} | corrected: {p_rf.mean():.4f} | actual: {va.is_fraud.mean():.4f}")
pd.DataFrame(board).T.sort_values("PR_AUC", ascending=False).round(3)

**✔ Checkpoint.** Random forest tops the board (PR-AUC ≈ 0.94, ≈ 89% of fraud inside the 0.5% budget). Logistic regression on the *same engineered features* is surprisingly close (≈ 0.93) — good features matter more than the algorithm. Corrected mean probability ≈ actual rate.

## Step 6 · Which features drive the forest? Impurity vs permutation importance

**Why this matters.** Impurity importance is computed on training data and favours high-cardinality continuous features. **Permutation importance** measures how much *validation* PR-AUC drops when a feature is shuffled — slower but more trustworthy, and model-agnostic.

#### ✏️ What to write in the next cell
- **Blank 10** *(inline)* — shuffle each column, measure the PR-AUC drop

In [ ]:
from sklearn.inspection import permutation_importance
imp = pd.Series(rf.named_steps["rf"].feature_importances_, index=FEATS).sort_values()
va_s = pd.concat([va[va.is_fraud == 1], va[va.is_fraud == 0].sample(40_000, random_state=RNG)])
perm = ____  # ✏️ B10
perm_s = pd.Series(perm.importances_mean, index=FEATS).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(13, 5)); imp.tail(12).plot.barh(ax=ax[0], title="impurity (train)"); perm_s.tail(12).plot.barh(ax=ax[1], color="tab:orange", title="permutation (validation PR-AUC drop)")
plt.tight_layout(); plt.show()

**✔ Checkpoint.** Both rank velocity features (`secs_since_prev`, `txn_count_1h`, `amount_vs_card_mean`) and device/3DS signals near the top; the orders differ in the tail.

## Step 7 · Test-month evaluation and hand-off

**Why this matters.** September is the untouched test month. We report the metrics the Fraud Ops head cares about, then save the feature table (Day 4 reads it), the forest and the metrics.

In [ ]:
import joblib, json
p_te = rf.predict_proba(te[FEATS])[:, 1]
m_te = at_budget(te.is_fraud, p_te, te.amount_aed)
days = te.txn_ts.dt.normalize().nunique()
print({k: round(v, 3) for k, v in m_te.items()}, f"| alerts/day ≈ {int(len(te) * 0.005 / days)}")
keep = ["txn_id", "txn_ts", "card_id", "customer_id", "amount_aed", "merchant_country", "is_fraud", "split"] + FEATS
tf[keep].to_parquet(CKPT / "card_features.parquet", index=False)
joblib.dump(rf, CKPT / "fraud_rf.joblib", compress=3)
(CKPT / "fraud_metrics.json").write_text(json.dumps({"valid": {k: float(v) for k, v in board["random forest"].items()},
    "test": {k: float(v) for k, v in m_te.items()}, "features": FEATS, "undersample_beta": beta, "budget": 0.005}, indent=2))
print("saved card_features.parquet, fraud_rf.joblib, fraud_metrics.json")

**✔ Checkpoint.** Recall and AED share stay ≈ 93–94% on September; precision@0.5% is lower (≈ 65%) because September has fewer frauds to fill the fixed budget. ≈ 40 alerts/day for the review team.

## ✅ Lab complete
The fraud model ranks 0.5% of traffic containing most of the fraud. **Tomorrow:** SVMs and neural networks — can they do better?